# 04_plink_analysis — Annotate epistasis results (FLU / PULV)

Input: PLINK `*.epi.qt`  
Output: annotated TSV with genomic positions + gene annotations per SNP.

In [19]:
from pathlib import Path

DRUG = "FLU"        # "FLU" or "PULV"
RUN_LABEL = "qtl"    # qtl, all, etc.

BASE_DIR = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis")
DATA_DIR = BASE_DIR / "data"

EPI_QT = DATA_DIR / f"epi_{DRUG.lower()}_{RUN_LABEL}.epi.qt"
SNP_INFO_TSV = DATA_DIR / "snp_info_complete.tsv"

OUT_TSV = DATA_DIR / f"epi_{DRUG.lower()}_{RUN_LABEL}_annotated.tsv"

print("EPI_QT:", EPI_QT)
print("SNP_INFO_TSV:", SNP_INFO_TSV)
print("OUT_TSV:", OUT_TSV)

EPI_QT: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/epi_flu_qtl.epi.qt
SNP_INFO_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/snp_info_complete.tsv
OUT_TSV: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/epi_flu_qtl_annotated.tsv


## 2) Load PLINK epistasis output

In [20]:
import pandas as pd

epi = pd.read_csv(EPI_QT, sep=r"\s+")
print("Epistasis table shape:", epi.shape)
print("Columns:", list(epi.columns))
display(epi.head())

Epistasis table shape: (21325, 7)
Columns: ['CHR1', 'SNP1', 'CHR2', 'SNP2', 'BETA_INT', 'STAT', 'P']


,CHR1,SNP1,CHR2,SNP2,BETA_INT,STAT,P
0,5,snp10459,7,snp14392,-0.006083,28.8020,8.035000e-08
1,5,snp10459,7,snp14393,-0.006106,29.0187,7.184000e-08
2,5,snp10459,7,snp14394,-0.006078,28.7570,8.223000e-08
3,5,snp10459,7,snp14395,-0.006168,29.6159,5.279000e-08
4,5,snp10459,7,snp14396,-0.006168,29.6159,5.279000e-08


## 3) Load SNP metadata and build SNP→annotation table

In [21]:
snp = pd.read_csv(SNP_INFO_TSV, sep="\t")
print("SNP metadata shape:", snp.shape)
display(snp.head())

# Minimal columns we want if present
want_cols = [
    "SNP",
    "Chromosome",
    "Position (bp)",
    "gene_name",
    "gene_standard_name",
]
have_cols = [c for c in want_cols if c in snp.columns]
snp_small = snp[have_cols].copy()

# Ensure SNP is string
snp_small["SNP"] = snp_small["SNP"].astype(str)

print("Using SNP annotation columns:", have_cols)
display(snp_small.head())

SNP metadata shape: (41594, 12)


,Unnamed: 0,Index,SNP,Chromosome,Position (bp),BY allele,RM allele,feature_type,gene_name,gene_chr_start,gene_chr_end,gene_standard_name
0,0,0,snp1,1,27210,A,G,genic,YAL063C,24000,27968,FLO9
1,1,1,snp2,1,27290,C,A,genic,YAL063C,24000,27968,FLO9
2,2,2,snp3,1,27356,T,C,genic,YAL063C,24000,27968,FLO9
3,3,3,snp4,1,27357,A,G,genic,YAL063C,24000,27968,FLO9
4,4,4,snp5,1,27370,G,A,genic,YAL063C,24000,27968,FLO9


Using SNP annotation columns: ['SNP', 'Chromosome', 'Position (bp)', 'gene_name', 'gene_standard_name']


,SNP,Chromosome,Position (bp),gene_name,gene_standard_name
0,snp1,1,27210,YAL063C,FLO9
1,snp2,1,27290,YAL063C,FLO9
2,snp3,1,27356,YAL063C,FLO9
3,snp4,1,27357,YAL063C,FLO9
4,snp5,1,27370,YAL063C,FLO9


## 4) Annotate SNP1 and SNP2 (position + gene columns)

In [22]:
# Merge for SNP1
epi_annot = epi.merge(
    snp_small.add_prefix("SNP1_"),
    left_on="SNP1",
    right_on="SNP1_SNP",
    how="left"
)

# Merge for SNP2
epi_annot = epi_annot.merge(
    snp_small.add_prefix("SNP2_"),
    left_on="SNP2",
    right_on="SNP2_SNP",
    how="left"
)

# Clean up duplicate join keys if present
drop_cols = [c for c in ["SNP1_SNP", "SNP2_SNP"] if c in epi_annot.columns]
epi_annot = epi_annot.drop(columns=drop_cols, errors="ignore")

print("Annotated table shape:", epi_annot.shape)
display(epi_annot.head())

Annotated table shape: (21325, 15)


,CHR1,SNP1,CHR2,SNP2,BETA_INT,STAT,P,SNP1_Chromosome,SNP1_Position (bp),SNP1_gene_name,SNP1_gene_standard_name,SNP2_Chromosome,SNP2_Position (bp),SNP2_gene_name,SNP2_gene_standard_name
0,5,snp10459,7,snp14392,-0.006083,28.8020,8.035000e-08,5,367674,YER105C,NUP157,7,461612,YGL016W,KAP122
1,5,snp10459,7,snp14393,-0.006106,29.0187,7.184000e-08,5,367674,YER105C,NUP157,7,461695,YGL016W,KAP122
2,5,snp10459,7,snp14394,-0.006078,28.7570,8.223000e-08,5,367674,YER105C,NUP157,7,461875,YGL016W,KAP122
3,5,snp10459,7,snp14395,-0.006168,29.6159,5.279000e-08,5,367674,YER105C,NUP157,7,462190,YGL016W,KAP122
4,5,snp10459,7,snp14396,-0.006168,29.6159,5.279000e-08,5,367674,YER105C,NUP157,7,462196,YGL016W,KAP122


## 5) Sanity checks: annotation coverage

In [23]:
def frac_notna(df, col):
    return df[col].notna().mean() if col in df.columns else None

for col in ["SNP1_Chromosome", "SNP1_Position (bp)", "SNP2_Chromosome", "SNP2_Position (bp)"]:
    if col in epi_annot.columns:
        print(f"{col}: {frac_notna(epi_annot, col):.3f} non-missing")

# Check P-value range
print("P min:", epi_annot["P"].min(), "P max:", epi_annot["P"].max())

SNP1_Chromosome: 1.000 non-missing
SNP1_Position (bp): 1.000 non-missing
SNP2_Chromosome: 1.000 non-missing
SNP2_Position (bp): 1.000 non-missing
P min: 4.622e-133 P max: 0.0001


## 6) Save annotated results

In [24]:
epi_annot.to_csv(OUT_TSV, sep="\t", index=False)
print("Saved:", OUT_TSV)

Saved: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/epi_flu_qtl_annotated.tsv
